
# XGBoost Regression: Manual Walkthrough (3 Stage, Multi-Feature)

This Colab notebook provides a **manual simulation** of how XGBoost works under the hood:
- Multi-feature toy dataset
- Residual correction across 3 stages
- Manual decision tree building and prediction

We simulate boosting using decision stumps (one-level trees), similar to XGBoost's gradient boosting approach.

---  


In [ ]:

import pandas as pd
import numpy as np

# Stage 0: Define a toy dataset with multiple features
data = pd.DataFrame({
    'CGPA': [6.7, 9.0, 7.5, 8.0],
    'X12_marks': [75, 95, 80, 85],
    'package': [4.5, 11.0, 6.0, 8.0]
})

data


In [ ]:

# Stage 1: Baseline model uses mean of the output (as initial guess)
baseline_prediction = data['package'].mean()
data['pred_stage1'] = baseline_prediction

# Residuals = actual - predicted
data['resid_stage1'] = data['package'] - data['pred_stage1']
data


In [ ]:

# Stage 2: Build a decision stump manually using CGPA
# Let's split at CGPA < 8.0

def tree_stage2(cgpa):
    return -1.5 if cgpa < 8.0 else 1.0

data['tree_output_2'] = data['CGPA'].apply(tree_stage2)

# Apply learning rate (eta)
eta = 0.3
data['pred_stage2'] = data['pred_stage1'] + eta * data['tree_output_2']

# Residuals for next stage
data['resid_stage2'] = data['package'] - data['pred_stage2']
data


In [ ]:

# Stage 3: Build a decision stump using 12th marks
# Let's split at marks < 85

def tree_stage3(x):
    return -1.0 if x < 85 else 0.5

data['tree_output_3'] = data['X12_marks'].apply(tree_stage3)
data['pred_stage3'] = data['pred_stage2'] + eta * data['tree_output_3']

# Final residuals
data['resid_stage3'] = data['package'] - data['pred_stage3']
data



### Final Stage Prediction
Now we compute predictions after 3 boosting stages and compare with the actual package values.


In [ ]:

final_predictions = data[['CGPA', 'X12_marks', 'package', 'pred_stage3', 'resid_stage3']]
final_predictions.rename(columns={'pred_stage3': 'final_pred'}, inplace=True)
final_predictions
